# Stock Trading with Reinforcement Learning

This notebook trains DQN, PPO, and A2C agents to trade stocks using technical indicators as observations and compares their performance.

**Sections:**
1. Installation & Imports
2. Data Preprocessing & Visualization
3. Reward Models (Statistical, LSTM, Linear NN, LLM)
4. Training (DQN, PPO, A2C)
5. Testing & Comparison
6. Policy Inspection
7. Real-time Data & News

## 1. Installation

In [ ]:
import sys, os

# Ensure the project root is on the path so `src` is importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Install dependencies (uncomment if running for the first time)
# !pip install -r ../requirements.txt

## 2. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from stable_baselines3 import DQN, PPO, A2C
from stable_baselines3.common.env_util import make_vec_env

from src.preprocessing import (
    add_technical_indicators, download_stock, flatten_columns, ALL_STOCKS,
    CONSUMER_DISCRETIONARY, CONSUMER_STAPLES, HEALTH_CARE,
    INDUSTRIAL, TECHNOLOGY, TELECOMMUNICATIONS, UTILITIES,
)
from src.environments import StockTradingEnv, StockTestEnv
from src.reward_models import (
    LSTMRewardPredictor, LinearRewardPredictor,
    train_model, prepare_linear_data, prepare_lstm_data,
)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Stocks universe: {len(ALL_STOCKS)} symbols')

## 3. Data Preprocessing & Visualization

In [ ]:
# Download sample data
temp_data = download_stock('AMZN', '2020-01-01', '2021-01-01')
temp_data.head()

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(18, 4))

axs[0].plot(temp_data['Close'], label='Close')
axs[0].plot(temp_data['SMA'], label='SMA')
axs[0].plot(temp_data['RSI'], label='RSI')
axs[0].legend()
axs[0].set_title('Price & Indicators')

axs[1].plot(temp_data['ATR_14'], label='ATR-14', color='darkorange')
axs[1].legend()
axs[1].set_title('Average True Range')

axs[2].plot(temp_data['CCI_20'], label='CCI-20', color='forestgreen')
axs[2].legend()
axs[2].set_title('Commodity Channel Index')

bar_width = 0.35
positions = np.arange(len(temp_data.index))
axs[3].bar(positions - bar_width/2, temp_data['Volume'], bar_width, color='skyblue', label='Volume')
axs[3].bar(positions + bar_width/2, temp_data['OBV'], bar_width, color='orange', label='OBV')
axs[3].legend()
axs[3].set_title('Volume vs OBV')

plt.tight_layout()
plt.show()

In [ ]:
print('Stock universe:')
print(ALL_STOCKS)

## 4. Reward Models

### 4a. LLM Reward Function (Optional — requires OPENAI_API_KEY)

Set your API key as an environment variable before running:
```bash
export OPENAI_API_KEY="sk-..."
```

In [ ]:
# Uncomment to test the LLM reward function
# from src.reward_models import get_llm_reward
# obs_sample = torch.rand((1, 10)).tolist()[0]
# print(get_llm_reward(action=0, obs=obs_sample))

### 4b. Train LSTM Reward Predictor

In [ ]:
lstm_train_loader, lstm_test_loader, lstm_scaler = prepare_lstm_data(
    ALL_STOCKS, '2020-01-01', '2021-01-01', sequence_length=10, batch_size=32
)
print(f'LSTM training batches: {len(lstm_train_loader)}')

In [ ]:
lstm_model = LSTMRewardPredictor(input_size=6, hidden_layer_size=64, output_size=1)
lstm_criterion = nn.MSELoss()
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)

lstm_losses = train_model(
    lstm_model, lstm_train_loader, lstm_criterion, lstm_optimizer,
    num_epochs=100, print_every=25
)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(lstm_losses[2:], label='LSTM Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('LSTM Reward Predictor — Training Loss')
plt.legend()
plt.show()

### 4c. Train Linear (NN) Reward Predictor

In [ ]:
linear_train_loader, linear_test_loader, linear_scaler = prepare_linear_data(
    ALL_STOCKS, '2020-01-01', '2021-01-01', batch_size=32
)
print(f'Linear training batches: {len(linear_train_loader)}')

In [ ]:
linear_model = LinearRewardPredictor(input_size=6)
linear_criterion = nn.MSELoss()
linear_optimizer = optim.Adam(linear_model.parameters(), lr=0.0001)

# NOTE: Original used 10000 epochs. Reduce if you want faster iteration.
linear_losses = train_model(
    linear_model, linear_train_loader, linear_criterion, linear_optimizer,
    num_epochs=1000, print_every=100
)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(linear_losses[10:], label='Linear NN Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Linear Reward Predictor — Training Loss')
plt.legend()
plt.show()

In [ ]:
# Quick inference test
linear_model.eval()
with torch.no_grad():
    sample = torch.rand((1, 6))
    print(f'Random input: {sample}')
    print(f'Model output: {linear_model(sample)}')

## 5. Create Training Environment

In [ ]:
env = StockTradingEnv(
    stock_symbols=ALL_STOCKS,
    window_size=10,
    start_date='2020-01-01',
    end_date='2021-01-01',
    render_mode='human',
)
print(f'Action space: {env.action_space}')
print(f'Observation space: {env.observation_space.shape}')

### 5a. Random Agent Baseline

In [ ]:
obs, info = env.reset()
done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

plt.plot(env.portfolio_history, label=env.current_symbol)
plt.xlabel('Step')
plt.ylabel('Portfolio Value ($)')
plt.title('Random Agent')
plt.legend()
plt.show()

## 6. Train RL Agents

In [ ]:
# Wrap in vectorised env for stable-baselines3
vec_env = make_vec_env(
    lambda: StockTradingEnv(ALL_STOCKS, 10, '2020-01-01', '2021-01-01'),
    n_envs=1,
)

### 6a. DQN

In [ ]:
dqn_model = DQN(
    'MlpPolicy', vec_env, verbose=1,
    learning_rate=1e-3, learning_starts=1000, batch_size=32,
    tau=1.0, gamma=0.99, train_freq=4, gradient_steps=1,
    target_update_interval=500,
    exploration_fraction=0.1, exploration_initial_eps=1.0, exploration_final_eps=0.6,
    max_grad_norm=10,
    tensorboard_log='./logs/dqn/',
)
dqn_model.learn(total_timesteps=10_000)
dqn_model.save('../models/dqn_stock')
print('DQN saved.')

### 6b. PPO

In [ ]:
ppo_model = PPO(
    'MlpPolicy', vec_env, verbose=1,
    learning_rate=1e-3, n_steps=2048, batch_size=64, n_epochs=10,
    gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.0,
    max_grad_norm=10,
    tensorboard_log='./logs/ppo/',
)
ppo_model.learn(total_timesteps=50_000)
ppo_model.save('../models/ppo_stock')
print('PPO saved.')

### 6c. A2C

In [ ]:
a2c_model = A2C(
    'MlpPolicy', vec_env, verbose=1,
    learning_rate=1e-3, n_steps=5, gamma=0.99, gae_lambda=1.0,
    ent_coef=0.0, vf_coef=0.5, max_grad_norm=0.5,
    use_rms_prop=True, normalize_advantage=False,
    tensorboard_log='./logs/a2c/',
)
a2c_model.learn(total_timesteps=50_000)
a2c_model.save('../models/a2c_stock')
print('A2C saved.')

## 7. Testing & Comparison

In [ ]:
# (Optional) Load saved models instead of retraining
# dqn_model = DQN.load('../models/dqn_stock')
# ppo_model = PPO.load('../models/ppo_stock')
# a2c_model = A2C.load('../models/a2c_stock')

In [ ]:
def evaluate_agent(model, symbol, start='2020-01-01', end='2021-01-01'):
    """Run a trained agent on a specific symbol. Returns (portfolio_history, cumulative_rewards)."""
    test_env = StockTestEnv(
        ALL_STOCKS, 10, start, end,
        render_mode=None, specific_symbol=symbol
    )
    obs, _ = test_env.reset()
    done = False
    cum_reward = 0.0
    cum_rewards = [0.0]
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = test_env.step(action)
        cum_reward += reward
        cum_rewards.append(cum_reward)
        done = terminated or truncated
    return test_env.portfolio_history, cum_rewards

In [ ]:
test_symbols = ['AAPL', 'NVAX', 'AMZN', 'GE', 'TSLA', 'INTC']

results = {}
for sym in test_symbols:
    print(f'Evaluating {sym}...')
    results[sym] = {
        'A2C': evaluate_agent(a2c_model, sym),
        'PPO': evaluate_agent(ppo_model, sym),
        'DQN': evaluate_agent(dqn_model, sym),
    }
print('Done.')

### Portfolio Value Comparison

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 12))
axs = axs.flatten()

for i, sym in enumerate(test_symbols):
    for agent_name in ['A2C', 'PPO', 'DQN']:
        portfolio_hist = results[sym][agent_name][0]
        axs[i].plot(portfolio_hist, label=agent_name)
    axs[i].set_title(sym)
    axs[i].set_xlabel('Step')
    axs[i].set_ylabel('Portfolio Value ($)')
    axs[i].legend()

plt.tight_layout()
plt.show()

### Cumulative Reward Comparison

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 12))
axs = axs.flatten()

for i, sym in enumerate(test_symbols):
    for agent_name in ['A2C', 'PPO', 'DQN']:
        cum_rewards = results[sym][agent_name][1]
        axs[i].plot(cum_rewards, label=agent_name)
    axs[i].set_title(sym)
    axs[i].set_xlabel('Step')
    axs[i].set_ylabel('Cumulative Reward')
    axs[i].legend()

plt.tight_layout()
plt.show()

### Final Portfolio ROI

In [ ]:
print(f'{"Symbol":<8} {"DQN":>10} {"PPO":>10} {"A2C":>10}')
print('-' * 40)
for sym in test_symbols:
    dqn_roi = results[sym]['DQN'][0][-1] / 10000 if results[sym]['DQN'][0] else 0
    ppo_roi = results[sym]['PPO'][0][-1] / 10000 if results[sym]['PPO'][0] else 0
    a2c_roi = results[sym]['A2C'][0][-1] / 10000 if results[sym]['A2C'][0] else 0
    print(f'{sym:<8} {dqn_roi:>10.4f} {ppo_roi:>10.4f} {a2c_roi:>10.4f}')

## 8. Policy Inspection

In [ ]:
# Example: inspect A2C policy output on a sample observation
sample_env = StockTradingEnv(ALL_STOCKS, 10, '2020-01-01', '2021-01-01')
obs, _ = sample_env.reset()
obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)

a2c_policy = a2c_model.policy
a2c_policy.eval()

with torch.no_grad():
    action_dist = a2c_policy.get_distribution(obs_tensor)
    probs = action_dist.distribution.probs
    print(f'Action probabilities: Buy={probs[0][0]:.3f}, Sell={probs[0][1]:.3f}, Hold={probs[0][2]:.3f}')

## 9. Real-time Data (Optional)

In [ ]:
import yfinance as yf

ticker = yf.Ticker('AMZN')
info = ticker.info
current_price = info.get('currentPrice') or info.get('regularMarketPrice')
print(f'AMZN current price: ${current_price}')

### Real-time Price Tracker (runs in a loop — interrupt kernel to stop)

In [ ]:
# Uncomment to run the live price tracker
# from IPython.display import clear_output
# import time
#
# prices = []
# ticker_symbol = 'AMZN'
#
# for _ in range(30):  # 30 updates, then stop
#     ticker = yf.Ticker(ticker_symbol)
#     price = ticker.info.get('currentPrice') or ticker.info.get('regularMarketPrice')
#     if price:
#         prices.append(price)
#     clear_output(wait=True)
#     plt.figure(figsize=(10, 6))
#     plt.plot(prices, label=ticker_symbol, marker='o')
#     plt.legend()
#     plt.xlabel('Update #')
#     plt.ylabel('Price ($)')
#     plt.title(f'{ticker_symbol}: ${price}')
#     plt.grid(True, linestyle='--', linewidth=0.5, color='gray')
#     plt.show()
#     time.sleep(5)

## 10. Real-time News

In [ ]:
import feedparser

SEARCH_TERM = 'AMZN'
rss_url = f'https://news.google.com/rss/search?q={SEARCH_TERM}&hl=en-US&gl=US&ceid=US:en'

feed = feedparser.parse(rss_url)
for entry in feed.entries[:10]:
    print(f'• {entry.title}')